In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
!bash load_data.sh

In [9]:
from pathlib import Path
import pandas as pd
import torch

DATA_DIR = Path("data")
TORONTO = DATA_DIR / Path("toronto")
TIMIT = DATA_DIR / Path("timit")

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

## EDA

In [ ]:
import librosa

toronto_sr = pd.Series(
    [librosa.get_samplerate(f) for f in TORONTO.rglob("*.wav")]
).value_counts()

timit_sr = pd.Series(
    [librosa.get_samplerate(f) for f in TIMIT.rglob("*.wav")]
).value_counts()

print(toronto_sr) # 44100
print(timit_sr) # 16000


44100    18303
Name: count, dtype: int64
16000    6300
Name: count, dtype: int64


In [14]:
TORONTO_SR = 44100
TIMIT_SR = 16000

In [34]:
labels = pd.read_json(TORONTO / "labels.jsonl", lines=True)

labels["dataset/toronto_0/toronto_0_0.wav"][0]

'Терміново! Загроза нового вторгнення в Україну нині дуже висока.'

In [38]:
toronto_ids = [f.name for f in TORONTO.glob("toronto_*")]

toronto_test_ids = set(['toronto_27', 'toronto_46', 'toronto_42', 'toronto_37', 'toronto_89', 'toronto_43',
'toronto_157', 'toronto_9', 'toronto_156', 'toronto_7', 'toronto_123', 'toronto_54', 'toronto_67',
'toronto_62', 'toronto_81', 'toronto_134', 'toronto_148', 'toronto_21', 'toronto_135', 'toronto_166',
'toronto_58'])

toronto_train_ids = set(toronto_ids) - set(toronto_test_ids)
print(f"intersection is 0: {len(toronto_test_ids & toronto_train_ids) == 0}")

toronto_train_df, toronto_test_df = [], []
for f in TORONTO.rglob("*.wav"):
    item = {
        "filename": f.name,
        "speaker_id": f.parent.name,
        "duration": librosa.get_duration(path=f, sr=TORONTO_SR),
        "label": labels[f"dataset/{f.parent.name}/{f.name}"][0]
    }
    
    if f.parent.name in toronto_train_ids:
        toronto_train_df.append(item)
    elif f.parent.name in toronto_test_ids:
        toronto_test_df.append(item)
    


intersection is 0: True
